# Week 5 Exercise – WikiQA RAG Assistant (abdussamadbello)

This notebook implements a small **WikiQA-based Retrieval-Augmented Generation (RAG) assistant**. It uses the `wiki_qa` dataset as a knowledge base, retrieves relevant answer sentences, and asks a frontier model to answer questions **with explanations and explicit evidence**.

In [ ]:
import os

from datasets import load_dataset
from dotenv import load_dotenv
from openai import OpenAI

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

load_dotenv(override=True)

# LLM client (frontier model)
client = OpenAI()
LLM_MODEL = "gpt-4.1-nano"

# Embeddings + vector store config
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DB_DIR = "week5_wikiqa_db"

# 1) Load WikiQA dataset and build Documents from answer sentences
ds = load_dataset("wiki_qa", split="train")

documents: list[Document] = []
for i, row in enumerate(ds):
    text = row["answer"]
    if not text:
        continue
    title = row.get("document_title", "unknown")
    documents.append(
        Document(
            page_content=text,
            metadata={
                "id": i,
                "title": title,
                "source": f"{title} #{i}",
                "orig_question": row["question"],
            },
        )
    )

# 2) Build embeddings and Chroma vector store
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=DB_DIR,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant answering questions using evidence from Wikipedia sentences.
You MUST:
- Answer the user's question concisely.
- Then explain the answer using the retrieved evidence.
- Explicitly quote or reference the evidence sentences by their numbers.
If the evidence does not support an answer, say you don't know.

Context:
{context}
"""


def answer_with_explanation(question: str):
    """Run retrieval and LLM to get an answer and explanation based on evidence."""
    docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[{i+1}] (title: {d.metadata.get('title')}) {d.page_content}"
        for i, d in enumerate(docs)
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    resp = client.chat.completions.create(model=LLM_MODEL, messages=messages)
    answer = resp.choices[0].message.content
    return answer, docs


def format_evidence(docs):
    """Format retrieved docs as markdown evidence block."""
    if not docs:
        return "_No evidence found_"
    lines = []
    for i, d in enumerate(docs, start=1):
        title = d.metadata.get("title", "unknown")
        lines.append(f"**[{i}] {title}**\n\n{d.page_content}\n")
    return "\n---\n".join(lines)

In [ ]:
import gradio as gr


def chat_fn(message, history):
    """Gradio chat handler: run RAG, update history and evidence."""
    answer, docs = answer_with_explanation(message)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    evidence_md = format_evidence(docs)
    return history, evidence_md


with gr.Blocks() as demo:
    gr.Markdown("# 🧠 WikiQA RAG Assistant\nAsk questions, see evidence + explanation.")
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(type="messages", height=500)
            msg = gr.Textbox(placeholder="Ask a Wikipedia-style question...")
        with gr.Column(scale=1):
            evidence_md = gr.Markdown("*Evidence will appear here*")

    msg.submit(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, evidence_md])

# To run the app locally, uncomment the line below:
# demo.launch()